<img src="causal-wizard-logo.png" width="60" align="left" style="margin-right: 14px;" />

# [Causal Wizard](https://causalwizard.app) &mdash; Results (2/2)

Run **01-identification-and-estimation.ipynb (1/2)** first to produce a `results.json`.
This notebook is pure presentation: every cell below reads from that file (plus your
original config and data, for the plots that need the raw rows) and renders one section
of the results report &mdash; no further statistical computation happens here.

**Run this from inside the repo's `notebooks/` directory** so the local `causalwizard`
package is found automatically; in Colab, the next cell installs it from GitHub.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("causalwizard") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "causalwizard @ git+https://github.com/drawlinson/causal_wizard_app.git#subdirectory=notebooks"],
        check=True,
    )

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from causalwizard import config, diagnostics, plotting, results_schema, display_utils
from causalwizard.counterfactuals import SCENARIOS

## Inputs

`results_path=None` auto-finds the most recently written `results*.json` in this folder - set it explicitly to load a different run instead. `config_path`/`data_path` aren't needed here: they're read straight out of the results file (notebook 1 saved them there).

In [ ]:
results_path = None  # None = auto-detect; or set e.g. "results-my-study.json"

results_path = results_path or results_schema.find_latest_results()
results = results_schema.load_results(results_path)
config_path = results["source"]["configPath"]
data_path = results["source"]["dataPath"]
print(f"Loaded {results_path} (config={config_path}, data={data_path})")

In [ ]:
cfg = config.load_config(config_path)
raw_df = config.align_columns(pd.read_csv(data_path), cfg)
prepared = config.prepare_dataframe(cfg, raw_df)
train_df, test_df = config.train_test_split(prepared.df, cfg["question"]["splitTestPc"])

q = results["question"]
e = results["estimate"]
outcome_is_binary = q["outcomeType"] == "categorical"

## Study summary

In [ ]:
display(Markdown(diagnostics.modelling_statements_markdown(results["modellingStatements"])))

## 1. Findings

In [ ]:
display(Markdown(diagnostics.findings_markdown(
    q["treatment"], q["outcome"], q["treatmentIsContinuous"], q["outcomeType"], q["outcomeClass1Label"],
    q["targetUnits"], e["effect"], e["accept"],
)))

## 2. Outcomes plots

Outcomes by Sample Cohort always renders. Outcomes by Entity (scatter of observed vs.
predicted against treatment value) only applies to a numerical/continuous treatment
design. Outcomes over Time only applies when the study has a panel-data time variable.

In [ ]:
plotting.plot_outcomes_cohort(train_df, q["treatment"], q["outcome"], outcome_is_binary).show()

tp = e["trainPredictions"]
if tp is None:
    print("This estimator has no do-operator, so no predicted-outcome plots are available.")
else:
    if q["treatmentIsContinuous"]:
        plotting.plot_outcomes_entity(
            train_df, q["treatment"], q["outcome"], None,
            predicted=tp["predicted"], control_pred=tp["counterfactualControl"], treated_pred=tp["counterfactualTreated"],
            control_value=e["counterfactualControlValue"], treated_value=e["counterfactualTreatedValue"],
        ).show()
    if q["panelData"] and q["panelData"].get("time"):
        plotting.plot_outcomes_over_time(
            train_df, q["panelData"]["time"], q["panelData"]["entity"], q["outcome"], tp["predicted"]
        ).show()

## 3. Counterfactual outcomes table

In [ ]:
cf = e["counterfactuals"]
available = not all(v is None for v in cf.values())
display(Markdown(diagnostics.counterfactual_markdown_intro(
    q["treatmentIsContinuous"], outcome_is_binary, e["counterfactualControlValue"], e["counterfactualTreatedValue"], available,
)))
if available:
    rows = [
        {"Scenario": label, **(cf[key] if cf[key] else {"count": None, "sum": None, "mean": None})}
        for key, label in SCENARIOS
    ]
    display(pd.DataFrame(rows).set_index("Scenario"))

## 4. Refutation / validation

In [ ]:
display(Markdown(diagnostics.validation_markdown_intro(e["validation"])))
rows = diagnostics.validation_rows(e["validation"])
if rows:
    display(pd.DataFrame(rows).set_index("Test"))

## 5. Held-out generalization

In [ ]:
gen = e["generalization"]
reason = diagnostics.generalization_unavailable_reason(q["method"], e["estimatorName"])
if gen is None:
    print(f"Not shown: {reason}" if reason else "Not available for this estimator.")
else:
    display(Markdown(diagnostics.generalization_markdown_intro(len(gen["actual"]))))
    actual, predicted = np.array(gen["actual"]), np.array(gen["predicted"])
    if outcome_is_binary:
        cm = plotting.confusion_matrix(gen)
        display(Markdown(diagnostics.generalization_metric_notes_markdown(True)))
        print(f"\nAccuracy={cm['accuracy']:.3f}  Precision={cm['precision']:.3f}  "
              f"Recall={cm['recall']:.3f}  F1={cm['f1']:.3f}")
        display_utils.show(cm, label="Confusion matrix")
    else:
        rmse = float(np.sqrt(np.mean((actual - predicted) ** 2)))
        mae = float(np.mean(np.abs(actual - predicted)))
        ss_res, ss_tot = float(np.sum((actual - predicted) ** 2)), float(np.sum((actual - actual.mean()) ** 2))
        r2 = 1 - ss_res / ss_tot if ss_tot else float("nan")
        display(Markdown(diagnostics.generalization_metric_notes_markdown(False)))
        print(f"\nR^2={r2:.3f}  RMSE={rmse:.4g}  MAE={mae:.4g}")
        display(Markdown(diagnostics.generalization_plot_markdown()))
        plotting.plot_generalization_scatter(gen, treatment_col_is_binary=not q["treatmentIsContinuous"]).show()

## 6. Contingency table

In [ ]:
display(Markdown(diagnostics.contingency_markdown_intro()))
if q["treatmentIsContinuous"]:
    print("Not applicable - this study uses a continuous treatment, with no Control/Treated split.")
else:
    ct = results["sample"]["contingencyTable"]
    display(pd.DataFrame([ct["total"]], index=["All outcomes"]))
    if "by_outcome" in ct:
        rows = [{"Outcome": k, "Control": v["control"], "Treated": v["treated"]} for k, v in ct["by_outcome"].items()]
        display(pd.DataFrame(rows).set_index("Outcome"))

## 7. Positivity check

In [ ]:
pa = e["propensityAnalysis"]
if pa is None:
    print("Not a propensity-based estimator.")
else:
    display(Markdown(diagnostics.positivity_markdown_intro()))
    plotting.plot_positivity(pa["distribution"]).show()

## 8. Covariate balance (love plot)

In [ ]:
if pa is None:
    print("Not a propensity-based estimator.")
else:
    display(Markdown(diagnostics.covariate_balance_markdown_intro()))
    plotting.plot_covariate_balance(pa["covariate_balance"]).show()

## 9. Summary results

In [ ]:
if e["regressionSummary"] is None:
    print("Not available for this estimator - only own linear regression/GLM and PD+FE fit a plain "
          "statsmodels regression with a full summary; propensity/DML/IV/frontdoor estimators don't.")
else:
    display(Markdown(diagnostics.regression_summary_markdown_intro()))
    display(Markdown(f"```\n{e['regressionSummary']}\n```"))

## 10. Assumptions

In [ ]:
display(Markdown(diagnostics.assumptions_markdown(q["method"], e["assumptions"])))

## 11. Causal diagram

CD+PO studies only.

In [ ]:
if q["method"] != "cd+po":
    print("Causal diagram only applies to CD+PO studies.")
else:
    display(Markdown(diagnostics.causal_diagram_markdown_intro()))
    roles = {q["treatment"]: "treatment", q["outcome"]: "outcome"}
    for var in e["estimandVariables"]:
        roles[var] = e["estimandType"]
    plotting.plot_causal_diagram(cfg["graph"], roles)